In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

In [2]:
import pandas as pd
from src.preprocessing import *

In [3]:
PATH = "../data/"
filename = 'msmes.csv'
df = load_data(PATH + filename)
df.head()

2026-07-07 20:36:54 - INFO - Loading dataset from ../data/msmes.csv


,Business_Size,Business_Age,Industry_Type,Entity_Type,Monthly_GST_Sales,GST_Growth_Rate,GST_Filing_Delay,GST_Compliance_Rate,Monthly_UPI_Count,Monthly_UPI_Value,...,Monthly_Credit,Monthly_Debit,Cashflow_Volatility,Employee_Count,Payroll_Consistency,Vendor_Payment_Delay,Working_Capital_Cycle,EMI_Bounce_Count,PD_True,Default
0,Small,3.259143,Retail,Proprietorship,1.041611e+06,-0.105973,11.100083,0.930616,89,222313.252027,...,1.079347e+06,799912.170455,0.016356,27,0.975329,11.376241,34.511462,0,0.000059,0
1,Micro,1.043744,Manufacturing,LLP,1.185538e+05,0.148953,8.625211,1.000000,13,5981.733679,...,1.101660e+05,92536.163891,0.341500,7,0.849837,19.329942,102.988904,0,0.001164,0
2,Micro,7.300253,Agriculture,Proprietorship,5.511186e+05,0.110490,4.305277,0.959657,17,17563.091000,...,4.778285e+05,279259.848766,0.527937,4,0.799129,20.347545,78.429116,1,0.103713,1
3,Micro,26.798767,Retail,Proprietorship,1.669279e+05,-0.015217,3.290730,1.000000,59,80650.048906,...,1.557148e+05,130123.809195,0.382392,5,0.706147,7.669705,43.484573,0,0.000008,0
4,Micro,6.507175,Manufacturing,Partnership,8.990894e+05,0.127848,3.903448,0.931603,14,31292.801883,...,9.385547e+05,755815.166491,0.174035,5,0.707897,19.509115,103.379616,1,0.007137,0


### Data Quality Report

In [4]:
report = inspect_data(df)

for key, value in report.items():
    print(f"{key}: {value}")
    print("=============================")

shape: (20000, 22)
columns: ['Business_Size', 'Business_Age', 'Industry_Type', 'Entity_Type', 'Monthly_GST_Sales', 'GST_Growth_Rate', 'GST_Filing_Delay', 'GST_Compliance_Rate', 'Monthly_UPI_Count', 'Monthly_UPI_Value', 'Digital_Sales_Ratio', 'Average_Bank_Balance', 'Monthly_Credit', 'Monthly_Debit', 'Cashflow_Volatility', 'Employee_Count', 'Payroll_Consistency', 'Vendor_Payment_Delay', 'Working_Capital_Cycle', 'EMI_Bounce_Count', 'PD_True', 'Default']
dtypes: {'Business_Size': <StringDtype(storage='python', na_value=nan)>, 'Business_Age': dtype('float64'), 'Industry_Type': <StringDtype(storage='python', na_value=nan)>, 'Entity_Type': <StringDtype(storage='python', na_value=nan)>, 'Monthly_GST_Sales': dtype('float64'), 'GST_Growth_Rate': dtype('float64'), 'GST_Filing_Delay': dtype('float64'), 'GST_Compliance_Rate': dtype('float64'), 'Monthly_UPI_Count': dtype('int64'), 'Monthly_UPI_Value': dtype('float64'), 'Digital_Sales_Ratio': dtype('float64'), 'Average_Bank_Balance': dtype('float64'

### Preprocessing

In [5]:
df = remove_target_leakage(df)
df = handle_missing_values(df)
df = detect_outliers(df)
df = engineer_features(df)

2026-07-07 20:36:54 - INFO - No missing values detected.
2026-07-07 20:36:54 - INFO - Synthetic dataset: outlier treatment skipped.
2026-07-07 20:36:54 - INFO - Using provided engineered features. -> skip for synthetic data


### Train-Test Split

In [6]:
train_df, test_df = create_train_test_split(df)

save_data(train_df, PATH+"train_msmes_data.csv")
save_data(test_df, PATH+"test_msmes_data.csv")

2026-07-07 20:36:55 - INFO - Dataset saved to ../data/train_msmes_data.csv
2026-07-07 20:36:55 - INFO - Dataset saved to ../data/test_msmes_data.csv


## Stats

In [7]:
import json

STATS_DICT = {}

# Numeric columns (excluding target)
numeric_cols = (
    train_df
    .drop(columns=["Default"])
    .select_dtypes(include="number")
    .columns
)

# Build statistics separately for each business size
for size in train_df["Business_Size"].unique():

    subset = train_df[train_df["Business_Size"] == size]

    STATS_DICT[size] = {}

    for feature in numeric_cols:

        STATS_DICT[size][feature] = {
            "p5": float(subset[feature].quantile(0.05)),
            "p95": float(subset[feature].quantile(0.95))
        }

In [8]:
import json
with open("../models/normalization_stats.json", "w") as file:
    json.dump(STATS_DICT, file, indent = 4)

In [9]:
STATS_DICT

{'Micro': {'Business_Age': {'p5': 1.0, 'p95': 17.793136580329975},
  'Monthly_GST_Sales': {'p5': 80717.46273265606, 'p95': 1595118.0248563525},
  'GST_Growth_Rate': {'p5': -0.09458522461259437, 'p95': 0.33402794236251787},
  'GST_Filing_Delay': {'p5': 1.580684110593177, 'p95': 27.24362331212065},
  'GST_Compliance_Rate': {'p5': 0.8194562191276434, 'p95': 1.0},
  'Monthly_UPI_Count': {'p5': 10.0, 'p95': 55.0},
  'Monthly_UPI_Value': {'p5': 9469.277170996387, 'p95': 101573.60118557833},
  'Digital_Sales_Ratio': {'p5': 0.3143490729628081, 'p95': 0.939758369546609},
  'Average_Bank_Balance': {'p5': 7036.427885772401, 'p95': 169649.18203578657},
  'Monthly_Credit': {'p5': 76301.71955019572, 'p95': 1502709.7459538947},
  'Monthly_Debit': {'p5': 56310.850241931184, 'p95': 1173218.878916322},
  'Cashflow_Volatility': {'p5': 0.08229262426309997,
   'p95': 0.6486497956428625},
  'Employee_Count': {'p5': 1.0, 'p95': 7.0},
  'Payroll_Consistency': {'p5': 0.6583727956021943, 'p95': 0.84049486371207